# Multiple Linear Regression End-to-End
Predict `sales_k_units` from numeric and categorical marketing drivers using scikit-learn pipelines and statsmodels OLS.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.formula.api as smf

DATA_DIR = Path.cwd().parent / 'data' if (Path.cwd().parent / 'data').exists() else Path.cwd() / 'data'
df = pd.read_csv(DATA_DIR / 'multiple_regression_marketing_sales.csv')
target = 'sales_k_units'
num = ['tv_spend_k','radio_spend_k','social_spend_k','price_index','competitor_spend_k']
cat = ['season']
X = df[num + cat]
y = df[target]
preprocess = ColumnTransformer([('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat), ('num', 'passthrough', num)])
pipe = Pipeline([('preprocess', preprocess), ('model', LinearRegression())])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=43)
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
print(f'rows: {len(df)}')
print(f'MAE: {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R2: {r2:.4f}')

rows: 120
MAE: 6.8034
RMSE: 9.0077
R2: 0.9597


In [2]:
features = pipe.named_steps['preprocess'].get_feature_names_out()
coefs = pipe.named_steps['model'].coef_
pd.DataFrame({'feature': features, 'coefficient': coefs}).sort_values('coefficient', ascending=False)

                    feature  coefficient
2            cat__season_Q4    16.074842
1            cat__season_Q3     9.514449
0            cat__season_Q2     9.473192
4        num__radio_spend_k     0.761870
3           num__tv_spend_k     0.456565
5       num__social_spend_k     0.376357
7  num__competitor_spend_k    -0.141192
6         num__price_index    -0.744473

In [3]:
ols = smf.ols('sales_k_units ~ tv_spend_k + radio_spend_k + social_spend_k + price_index + competitor_spend_k + C(season)', data=df).fit()
pd.DataFrame({'coefficient': ols.params, 'p_value': ols.pvalues}).sort_values('p_value')

                       coefficient       p_value
tv_spend_k                0.450606  1.983393e-73
radio_spend_k             0.803542  7.038510e-41
price_index              -0.749053  2.496985e-16
competitor_spend_k       -0.152480  6.154205e-14
social_spend_k            0.402834  4.656125e-12
C(season)[T.Q4]          16.733227  4.836192e-10
C(season)[T.Q2]          12.046460  1.357397e-06
C(season)[T.Q3]          11.770254  3.909291e-06
Intercept                39.157222  3.831908e-05

## Interpretation
Radio, TV, and social spend have positive coefficients. Price index and competitor spend have negative coefficients. Season coefficients are interpreted relative to the dropped baseline season.